In [1]:
import requests

session = requests.Session()
session.headers.update({'X-Student': 'miren'})

session.get('https://httpbin.org/cookies/set?course=data_science')
headers_resp = session.get('https://httpbin.org/headers').json()
cookies_resp = session.get('https://httpbin.org/cookies').json()

print('headers:', headers_resp.get('headers'))
print('cookies:', cookies_resp.get('cookies'))

headers: {'Accept': '*/*', 'Accept-Encoding': 'gzip, deflate', 'Host': 'httpbin.org', 'User-Agent': 'python-requests/2.32.3', 'X-Student': 'miren'}
cookies: {'course': 'data_science'}


In [2]:
import requests
import json

report = []

# 1. api key in header
r1 = requests.get('https://httpbin.org/headers', headers={'X-API-Key': 'key_98765'})
report.append({
    'call': 'api_key_header',
    'status': r1.status_code,
    'result': f"x-api-key={r1.json().get('headers', {}).get('X-Api-Key')}"
})

# 2. api key in query param
r2 = requests.get('https://httpbin.org/get', params={'api_key': 'key_98765'})
report.append({
    'call': 'api_key_query',
    'status': r2.status_code,
    'result': f"api_key={r2.json().get('args', {}).get('api_key')}"
})

# 3. bearer token
r3 = requests.get('https://httpbin.org/bearer', headers={'Authorization': 'Bearer test_token_123'})
report.append({
    'call': 'bearer_valid',
    'status': r3.status_code,
    'result': f"authenticated={r3.json().get('authenticated')}"
})

# 4. basic auth
r4 = requests.get('https://httpbin.org/basic-auth/student/pass123', auth=('student', 'pass123'))
report.append({
    'call': 'basic_auth',
    'status': r4.status_code,
    'result': f"authenticated={r4.json().get('authenticated')}, user={r4.json().get('user')}"
})

# 5. bearer missing header
r5 = requests.get('https://httpbin.org/bearer')
report.append({
    'call': 'bearer_missing',
    'status': r5.status_code,
    'result': 'unauthorized without bearer header'
})

with open('auth_report.json', 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2)

for item in report:
    print(item)

{'call': 'api_key_header', 'status': 200, 'result': 'x-api-key=key_98765'}
{'call': 'api_key_query', 'status': 200, 'result': 'api_key=key_98765'}
{'call': 'bearer_valid', 'status': 200, 'result': 'authenticated=True'}
{'call': 'basic_auth', 'status': 200, 'result': 'authenticated=True, user=student'}
{'call': 'bearer_missing', 'status': 401, 'result': 'unauthorized without bearer header'}


In [3]:
import asyncio
import time

async def fake_api_call(i):
    await asyncio.sleep(1)
    return i

async def main():
    t0 = time.perf_counter()
    seq_results = []
    for i in range(1, 6):
        res = await fake_api_call(i)
        seq_results.append(res)
    t1 = time.perf_counter()
    print(f'sequential: {t1 - t0:.2f}s', seq_results)

    t2 = time.perf_counter()
    conc_results = await asyncio.gather(*(fake_api_call(i) for i in range(1, 6)))
    t3 = time.perf_counter()
    print(f'concurrent: {t3 - t2:.2f}s', conc_results)

await main()

sequential: 5.00s [1, 2, 3, 4, 5]
concurrent: 1.00s [1, 2, 3, 4, 5]
